# Phase 1: Exploratory Data Analysis

**Goal:** Understand the dataset structure, the severity of class imbalance, and the relationships between features and the fraud label.  
**Dataset:** Kaggle Credit Card Fraud Detection — 284,807 transactions, 492 fraudulent (0.17%)

**Setup:** Download the dataset first:
```bash
kaggle datasets download -d mlg-ulb/creditcardfraud
unzip creditcardfraud.zip -d data/raw/
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load & Basic Inspection

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
# Check for nulls
print('Null counts:')
print(df.isnull().sum().sum(), 'total nulls')

## 2. Class Imbalance

This is the most important property of the dataset to understand before modeling.

In [ ]:
class_counts = df['Class'].value_counts()
fraud_rate = class_counts[1] / len(df) * 100

print(f'Legitimate transactions: {class_counts[0]:,}')
print(f'Fraudulent transactions:  {class_counts[1]:,}')
print(f'Fraud rate: {fraud_rate:.4f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Legitimate (0)', 'Fraud (1)'], class_counts.values, color=['steelblue', 'crimson'])
axes[0].set_title('Class Distribution (raw counts)')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center')

axes[1].pie(class_counts.values, labels=['Legitimate', 'Fraud'],
            colors=['steelblue', 'crimson'], autopct='%1.3f%%', startangle=90)
axes[1].set_title('Class Distribution (proportion)')

plt.tight_layout()
plt.savefig('../data/eda_class_distribution.png', dpi=120)
plt.show()

**Key insight:** A naive model that always predicts "legitimate" would achieve 99.83% accuracy — this is why **accuracy is a useless metric here**. We must focus on Precision, Recall, and especially **AUC-PR** (Area Under the Precision-Recall Curve).

## 3. Feature Distributions: Amount & Time

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Amount distribution
axes[0, 0].hist(df[df['Class'] == 0]['Amount'], bins=100, color='steelblue', alpha=0.7, label='Legitimate')
axes[0, 0].set_title('Transaction Amount — Legitimate')
axes[0, 0].set_xlabel('Amount (€)')
axes[0, 0].set_xlim(0, 2000)

axes[0, 1].hist(df[df['Class'] == 1]['Amount'], bins=50, color='crimson', alpha=0.7, label='Fraud')
axes[0, 1].set_title('Transaction Amount — Fraud')
axes[0, 1].set_xlabel('Amount (€)')

# Time distribution
axes[1, 0].hist(df[df['Class'] == 0]['Time'] / 3600, bins=100, color='steelblue', alpha=0.7)
axes[1, 0].set_title('Transaction Time — Legitimate')
axes[1, 0].set_xlabel('Hours since first transaction')

axes[1, 1].hist(df[df['Class'] == 1]['Time'] / 3600, bins=50, color='crimson', alpha=0.7)
axes[1, 1].set_title('Transaction Time — Fraud')
axes[1, 1].set_xlabel('Hours since first transaction')

plt.tight_layout()
plt.savefig('../data/eda_amount_time.png', dpi=120)
plt.show()

In [ ]:
print('Amount statistics by class:')
print(df.groupby('Class')['Amount'].describe())

## 4. V1–V28 Feature Distributions by Class

The V features are PCA-transformed for privacy, but we can still see which ones separate fraud from legitimate transactions.

In [ ]:
v_features = [f'V{i}' for i in range(1, 29)]

# Compare means: fraud vs legitimate for each V feature
means = df.groupby('Class')[v_features].mean().T
means.columns = ['Legitimate', 'Fraud']
means['diff'] = (means['Fraud'] - means['Legitimate']).abs()
means_sorted = means.sort_values('diff', ascending=False)

print('Top 10 features by mean difference (fraud vs legitimate):')
print(means_sorted.head(10))

In [ ]:
# Visualise top 6 most discriminative features
top_features = means_sorted.index[:6].tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, feat in zip(axes.flat, top_features):
    ax.hist(df[df['Class'] == 0][feat], bins=80, color='steelblue', alpha=0.6, density=True, label='Legitimate')
    ax.hist(df[df['Class'] == 1][feat], bins=80, color='crimson', alpha=0.6, density=True, label='Fraud')
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.suptitle('Top 6 Discriminative Features (density)', y=1.01)
plt.tight_layout()
plt.savefig('../data/eda_top_features.png', dpi=120)
plt.show()

## 5. Correlation Heatmap

In [ ]:
# Correlation with Class label
corr_with_class = df.corr()['Class'].drop('Class').sort_values(key=abs, ascending=False)
print('Features most correlated with Class (fraud):')
print(corr_with_class.head(15))

In [ ]:
plt.figure(figsize=(8, 6))
corr_with_class.head(15).sort_values().plot(kind='barh', color='crimson')
plt.title('Feature Correlation with Fraud Label (top 15)')
plt.xlabel('Pearson correlation')
plt.tight_layout()
plt.savefig('../data/eda_correlations.png', dpi=120)
plt.show()

## 6. Summary of EDA Findings

| Finding | Implication |
|---|---|
| Fraud rate = 0.17% | Accuracy is meaningless; use Precision, Recall, AUC-PR |
| V1–V28 already normalized | Only Amount and Time need scaling |
| V4, V11, V14, V17 most discriminative | Tree models will leverage these heavily |
| Fraud amounts not dramatically higher | Can't use a simple threshold on amount |
| No null values | No imputation needed |

**Next step → `02_baseline.ipynb`**: scale features, create stratified splits, train a logistic regression baseline.